In [1]:
"""
=============================================================
 BERT AI Classifier: CSV Batch Prediction
=============================================================
 This script loads the saved BERT model and processes an 
 input CSV (ID;Text) to produce an output CSV (ID;Label).
=============================================================
"""

import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# ── 1. Config ─────────────────────────────────────────────────────────────────
SAVE_DIR = "./bert-ai-classifier"  # The folder created by the training script
INPUT_FILE = "subm1.csv"      # The input file to be classified
OUTPUT_FILE = "predictions.csv"    # The file to be submitted
MAX_LEN = 256

# Auto-detect device
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

# ── 2. Load Model & Tokenizer ─────────────────────────────────────────────────
print(f"Loading model from {SAVE_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
model.to(DEVICE)
model.eval()

# ── 3. Processing Logic ───────────────────────────────────────────────────────
def run_predictions(input_path, output_path):
    # Load the CSV with the semicolon delimiter
    df = pd.read_csv(input_path, sep=';')
    
    text_col = next((c for c in df.columns if c.lower() == 'text'), None)
    id_col = next((c for c in df.columns if c.lower() == 'id'), None)

    if not text_col or not id_col:
        print(f"Error: Could not find 'ID' or 'Text' columns in {input_path}")
        return

    predictions = []

    print(f"Processing {len(df)} rows...")
    
    with torch.no_grad():
        for text in tqdm(df[text_col].astype(str).tolist()):
            inputs = tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(DEVICE)

            # Forward pass
            outputs = model(**inputs)
            
            # Get the highest probability index
            pred_idx = torch.argmax(outputs.logits, dim=1).item()
            
            # Map index back to string label (Human, OpenAI, etc.)
            label = model.config.id2label[pred_idx]
            predictions.append(label)

    # Add predictions to a new column
    df['Label'] = predictions

    # Create the final dataframe with only ID and Label
    output_df = df[[id_col, 'Label']]

    output_df.to_csv(output_path, sep=';', index=False)
    print(f"\nSuccessfully saved predictions to: {output_path}")

# ── 4. Execution ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import os
    if os.path.exists(INPUT_FILE):
        run_predictions(INPUT_FILE, OUTPUT_FILE)
    else:
        print(f"Error: File '{INPUT_FILE}' not found. Please check the filename.")

c:\Users\alex3\miniconda3\envs\AP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Loading model from ./bert-ai-classifier...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 17198.54it/s]


Processing 150 rows...


100%|██████████| 150/150 [00:01<00:00, 138.03it/s]


Successfully saved predictions to: predictions.csv
